# A+B+Diversifier Sleeves — Analytics Notebook

This notebook reconstructs the ensemble strategy and produces:
- Cumulative returns
- Key metrics (Total Return, CAGR, Sharpe, Sortino, Max DD, Win Rate, Profit Factor, PSR, DSR)
- Drawdown over time
- Daily returns distribution
- Monthly returns heatmap
- Rolling Sharpe (252 days)
- Rolling volatility (252 days)


In [ ]:
import sys
from pathlib import Path
from datetime import date, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# The kernel runs with cwd = this notebook's folder (Live/notebooks). The `live`
# package is one level up (Live/); the pre-computed A/B research CSVs are two
# levels up (Trading I/). Search upward so it also works if run from Live/ itself.
cwd = Path.cwd()
REPO_ROOT = cwd.parent if (cwd.parent / 'live' / '__init__.py').exists() else cwd
PARENT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from live.data_feed import fetch_panel
from live.portfolio import build_sleeve_returns, SleeveConfig

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print(f'Imports OK (REPO_ROOT={REPO_ROOT}, PARENT={PARENT})')


## 1. Load Strategy A / B returns and sleeve prices

In [ ]:
# Paths to pre-computed core returns (generated by research scripts in the parent folder).
CORE_A_CSV = PARENT / '_rsi_rotation_compare_strategy_A.csv'
CORE_B_CSV = PARENT / '_rsi_rotation_compare_strategy_B.csv'

ret_a = pd.read_csv(CORE_A_CSV, index_col='date', parse_dates=['date'])['strategy_return']
ret_b = pd.read_csv(CORE_B_CSV, index_col='date', parse_dates=['date'])['strategy_return']

common_index = ret_a.index.intersection(ret_b.index)
ret_a = ret_a.loc[common_index]
ret_b = ret_b.loc[common_index]

# Sleeve ETF universe.
sleeve_tickers = ['SPY','TLT','IEF','GLD','PDBC','KMLM','DBMF','VIXM','SH','BIL','VIXY','PSQ','^VIX']
end = date(2025, 12, 31)  # snapshot window, matches README §12 (2015-2025)
start = ret_a.index.min().date() - timedelta(days=365)

prices = fetch_panel(sleeve_tickers, start, end, prefer_alpaca=False)
prices = prices.rename(columns={'^VIX': 'VIX'})

print(f'Core A/B dates: {ret_a.index.min().date()} -> {ret_a.index.max().date()}')
print(f'Price panel shape: {prices.shape}')


## 2. Build ensemble return series

In [ ]:
config = SleeveConfig()
sleeve_rets = build_sleeve_returns(prices)
sleeve_rets = sleeve_rets.reindex(common_index)

# BIL ballast: under the live default disable_bear=True, the bear sleeve's 20%
# routes to cash (BIL), held constantly (~0 turnover). Same shift(1) machinery as
# the other sleeves, so this is the live-config backtest -- NOT the SH/BIL bear
# sleeve, which was strictly dominated by cash on both Sharpe and MaxDD.
from live.portfolio import _net_sleeve_return
bil_rets = prices.pct_change(fill_method=None)
bil_ballast = _net_sleeve_return(
    pd.DataFrame({'BIL': 1.0}, index=prices.index), bil_rets, 0.0
).reindex(common_index)

# Daily ensemble net return (fixed sleeve weights).
ensemble_ret = (
    config.weight_a * ret_a
    + config.weight_b * ret_b
    + config.weight_rates * sleeve_rets['rates']
    + config.weight_bear * bil_ballast   # disable_bear=True -> cash, not the SH/BIL bear sleeve
    + config.weight_cta * sleeve_rets['cta']
)
ensemble_ret = ensemble_ret.dropna()

cum = (1 + ensemble_ret).cumprod()
print(f'Ensemble dates: {ensemble_ret.index.min().date()} -> {ensemble_ret.index.max().date()}')

## 3. Key metrics

In [ ]:
def probabilistic_sharpe_ratio(returns, target_sharpe=0.0):
    """Bailey & López de Prado PSR. `target_sharpe` is per-observation (daily).

    ponytail: the z numerator must use the per-observation Sharpe (sr_daily), not the
    annualized one — sqrt(T-1) is in daily observations, so annualizing inflates z by
    ~sqrt(252) and pins PSR at ~1.0. denom_sq already uses sr_daily correctly.
    """
    r = returns.dropna()
    T = len(r)
    if T < 30:
        return np.nan
    mean, std = r.mean(), r.std(ddof=1)
    if std == 0:
        return np.nan
    sr_daily = mean / std
    skew = r.skew()
    kurt = r.kurtosis() + 3.0
    denom_sq = 1.0 - skew * sr_daily + ((kurt - 1.0) / 4.0) * (sr_daily ** 2)
    if denom_sq <= 0:
        return np.nan
    z = (sr_daily - target_sharpe) * np.sqrt(T - 1) / np.sqrt(denom_sq)
    return float(stats.norm.cdf(z))

def deflated_sharpe_ratio(returns, n_trials=10, annualized_sharpe=None):
    """Bailey & López de Prado deflated SR.

    DSR = Φ(z_psr - E[max over N trials]), where z_psr is the PSR z-stat at target 0
    (per-observation Sharpe) and expected_max is the standardized expected max of N
    iid standard normals (~1.40 for N=10) — i.e. SR_0 = σ_SR · expected_max, so the
    σ_SR cancels into z_psr and we subtract expected_max in standardized units.

    ponytail: the old code compared the ANNUALIZED Sharpe to this standardized
    expected_max and used annualized in the z numerator — mixing units and inflating
    z by ~sqrt(252). Use sr_daily throughout; expected_max is unitless (in σ_SR).
    """
    r = returns.dropna()
    T = len(r)
    if T < 30 or n_trials <= 0:
        return np.nan
    if annualized_sharpe is None:
        annualized_sharpe = r.mean() / r.std(ddof=1) * np.sqrt(252)
    sr_daily = annualized_sharpe / np.sqrt(252)
    skew = r.skew()
    kurt = r.kurtosis() + 3.0
    denom_sq = 1.0 - skew * sr_daily + ((kurt - 1.0) / 4.0) * (sr_daily ** 2)
    if denom_sq <= 0:
        return np.nan
    z_psr = sr_daily * np.sqrt(T - 1) / np.sqrt(denom_sq)
    expected_max = np.sqrt(-np.log(1 - 0.5 ** (1.0 / n_trials)) / np.log(4))
    return float(stats.norm.cdf(z_psr - expected_max))

def metrics_table(returns, n_trials=10):
    r = returns.dropna()
    years = len(r) / 252.0
    cum = (1 + r).cumprod()
    total = cum.iloc[-1] - 1
    cagr = (cum.iloc[-1]) ** (1 / years) - 1 if years > 0 else 0.0
    ann_vol = r.std() * np.sqrt(252)
    sharpe = (r.mean() * 252) / ann_vol if ann_vol > 0 else 0.0
    downside = r[r < 0]
    sortino = (r.mean() * 252) / (downside.std() * np.sqrt(252)) if len(downside) > 1 else 0.0
    peak = cum.cummax()
    max_dd = ((cum - peak) / peak).min()
    win_rate = (r > 0).mean()
    gross_profit = r[r > 0].sum()
    gross_loss = -r[r < 0].sum()
    profit_factor = gross_profit / gross_loss if gross_loss != 0 else np.nan
    psr = probabilistic_sharpe_ratio(r)
    dsr = deflated_sharpe_ratio(r, n_trials=n_trials, annualized_sharpe=sharpe)
    return pd.Series({
        'Total Return': f'{total*100:.2f}%',
        'CAGR': f'{cagr*100:.2f}%',
        'Ann Vol': f'{ann_vol*100:.2f}%',
        'Sharpe': f'{sharpe:.2f}',
        'Sortino': f'{sortino:.2f}',
        'Max DD': f'{max_dd*100:.2f}%',
        'Win Rate': f'{win_rate*100:.2f}%',
        'Profit Factor': f'{profit_factor:.2f}',
        'PSR': f'{psr:.2%}',
        'DSR': f'{dsr:.2%}',
    })

metrics = metrics_table(ensemble_ret)
is_ = metrics_table(ensemble_ret.loc[:pd.Timestamp('2019-12-31')])
oos = metrics_table(ensemble_ret.loc[pd.Timestamp('2020-01-01'):])
display(pd.DataFrame({'Full 2015-2025': metrics, 'IS 2015-2019': is_, 'OOS 2020-2025': oos}))

# Beta / alpha (CAPM, Jensen's, rf = BIL) vs SPY and 60/40 (SPY 60% / IEF 40%).
rf = prices['BIL'].pct_change(fill_method=None)
spy = prices['SPY'].pct_change(fill_method=None)
bench_6040 = 0.6*prices['SPY'].pct_change(fill_method=None) + 0.4*prices['IEF'].pct_change(fill_method=None)

def beta_alpha(strat, bench, rfree):
    df = pd.concat([strat, bench, rfree], axis=1, keys=['s','b','r']).dropna()
    es = df['s'] - df['r']; eb = df['b'] - df['r']
    beta = es.cov(eb) / eb.var() if eb.var() > 0 else float('nan')
    alpha = 252 * (es.mean() - beta * eb.mean())
    return beta, alpha, es.corr(eb)

for label, bench in [('SPY', spy), ('60/40 (SPY/IEF)', bench_6040)]:
    b, a, c = beta_alpha(ensemble_ret, bench, rf)
    print(f'vs {label}: beta {b:.2f}, alpha {a*100:.2f}%/yr, corr {c:.2f}')

## 4. Cumulative returns

In [ ]:
fig, ax = plt.subplots()
cum.plot(ax=ax, label='A+B+Diversifier Sleeves', linewidth=1.5)
ax.axhline(1.0, color='black', linestyle='--', linewidth=0.8)
ax.set_title('Cumulative Returns')
ax.set_ylabel('Growth of $1')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Drawdown over time

In [ ]:
peak = cum.cummax()
dd = (cum - peak) / peak

fig, ax = plt.subplots()
dd.plot(ax=ax, color='crimson', linewidth=1.0)
ax.fill_between(dd.index, dd, 0, color='crimson', alpha=0.3)
ax.set_title('Drawdown Over Time')
ax.set_ylabel('Drawdown')
ax.set_ylim(dd.min() * 1.1, 0.02)
plt.tight_layout()
plt.show()
print(f'Max drawdown: {dd.min()*100:.2f}% on {dd.idxmin().date()}')

## 6. Daily returns distribution

In [ ]:
fig, ax = plt.subplots()
ensemble_ret.hist(bins=100, ax=ax, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(ensemble_ret.mean(), color='red', linestyle='--', label=f'Mean={ensemble_ret.mean()*100:.3f}%')
ax.set_title('Daily Returns Distribution')
ax.set_xlabel('Daily Return')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Skew: {ensemble_ret.skew():.2f}, Kurtosis: {ensemble_ret.kurtosis():.2f}')

## 7. Monthly returns heatmap

In [ ]:
monthly = ensemble_ret.resample('ME').apply(lambda x: (1 + x).prod() - 1) * 100
monthly_table = monthly.to_frame('ret').assign(
    year=monthly.index.year,
    month=monthly.index.month
).pivot(index='year', columns='month', values='ret')

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(monthly_table, annot=True, fmt='.1f', cmap='RdYlGn', center=0, ax=ax, linewidths=0.5)
ax.set_title('Monthly Returns Heatmap (%)')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## 8. Rolling Sharpe (252 days)

In [ ]:
rolling_sharpe = (ensemble_ret.rolling(252).mean() * 252) / (ensemble_ret.rolling(252).std() * np.sqrt(252))

fig, ax = plt.subplots()
rolling_sharpe.plot(ax=ax, color='green', linewidth=1.0)
ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.set_title('Rolling 252-Day Sharpe Ratio')
ax.set_ylabel('Sharpe')
plt.tight_layout()
plt.show()
print(f'Median rolling Sharpe: {rolling_sharpe.median():.2f}')

## 9. Rolling volatility (252 days)

In [ ]:
rolling_vol = ensemble_ret.rolling(252).std() * np.sqrt(252) * 100

fig, ax = plt.subplots()
rolling_vol.plot(ax=ax, color='purple', linewidth=1.0)
ax.axhline(rolling_vol.mean(), color='black', linestyle='--', linewidth=0.8, label=f'Mean={rolling_vol.mean():.1f}%')
ax.set_title('Rolling 252-Day Annualized Volatility')
ax.set_ylabel('Volatility (%)')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Component sleeve cumulative returns

In [ ]:
components = pd.DataFrame({
    'A': ret_a.reindex(ensemble_ret.index),
    'B': ret_b.reindex(ensemble_ret.index),
    'Rates': sleeve_rets['rates'].reindex(ensemble_ret.index),
    'BIL ballast': bil_ballast.reindex(ensemble_ret.index),
    'CTA': sleeve_rets['cta'].reindex(ensemble_ret.index),
})
cum_components = (1 + components).cumprod()

fig, ax = plt.subplots(figsize=(14, 7))
cum_components.plot(ax=ax, linewidth=1.2)
ax.set_title('Cumulative Returns by Sleeve Component')
ax.set_ylabel('Growth of $1')
ax.legend()
plt.tight_layout()
plt.show()